# KernelCPD Parameter Grid: Precision/Recall/F1 vs. kernel, pen, min_size (red/snr3)

Single-method, single-condition diagnostic: fix `ruptures` algorithm to **KernelCPD**, run it on
the **red/snr3** noise condition only (`noise_sensitivity_data/noise_red_snr3.npz`, 20 seeds), and
grid-search `pen` x `min_size` for **2 of KernelCPD's 3 kernels** (`linear`, `rbf`). Mirrors
`pelt_grid_red_snr3.ipynb` / `binseg_grid_red_snr3.ipynb` / `bottomup_grid_red_snr3.ipynb` --
same data, same ground truth, same methodology. No `width` parameter (that's Window-only).

**Why only 2 of 3 kernels.** `KernelCPD` supports exactly `'linear'`, `'rbf'`, `'cosine'`
(`ruptures_transitions.py` enforces this). `cosine` is dropped here: for a *univariate* signal,
the cosine kernel between two scalar points reduces to `sign(x_i) * sign(x_j) in {-1, +1}` -- it's
degenerate unless the series crosses zero, so it carries essentially no information about mean
shifts in data like this (same standard of evidence as the `ar`/`linear`-cost zero-detection notes
in the other notebooks, just derived analytically instead of empirically).

**Note on `'linear'` here vs. `'linear'` elsewhere.** This is NOT the same object as the
`CostLinear` model used by Pelt/Binseg/BottomUp/Window's `cost='linear'` (which came back
all-zero everywhere -- it targets a regression relationship absent from this univariate data).
KernelCPD's linear *kernel* is mathematically equivalent to the L2 (mean-shift) cost, so it
should track the L2 results elsewhere (e.g. BottomUp: test_f1 approx 0.85) rather than the
CostLinear zero-detection pattern. Starting `pen` range below is seeded from the other notebooks'
L2 range accordingly -- watch for `on_edge` and widen if it fires.

**Each kernel has its own cells (Section 7), same as before** -- one to define its grid and run
it, one to plot it. The grid-search/evaluation logic itself is NOT duplicated per cell -- every
kernel's cell just calls the same shared `run_one_cost(...)` function defined once in Section 5,
passing its own `pens`/`min_sizes`.

**Train/test split.** Seeds are split ONCE into a train pool and a test pool:

1. Grid search runs on the **train pool only** -- this produces the 3D "optimization surface"
   plot you see immediately after running each kernel's cell, and the winning `(pen, min_size)` is
   whichever grid point has the best mean f1 *there*.
2. That single winning combo is then applied to the **test pool** exactly once. Those numbers
   (`test_f1`, `test_precision`, `test_recall`) are the honest, held-out result.
3. A precision/recall tradeoff curve is also swept on the test pool (`pen` varied at the winning
   `min_size`) -- the closest analog to a PR curve available here.

No inner K-fold CV, same rationale as the other notebooks: `pen`/`min_size` are applied directly
per seed, never fit, so K-fold would mathematically collapse to the flat training-pool mean.

**Seed budget.** `rbf` is ~100-1000x slower per call than `linear` (true for KernelCPD same as
for the cost-function algorithms), so it gets the smaller seed budget (10 seeds: 7 train / 3
test). `linear` uses the full budget (20 seeds: 14 train / 6 test).


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ammonyte as amt
from ammonyte.utils.metrics import evaluate_detection
from joblib import Parallel, delayed
from mpl_toolkits.mplot3d import Axes3D

## Section 1 -- Configuration

In [ ]:
DATA_PATH   = "../../noise_sensitivity_data/noise_red_snr3.npz"
TOLERANCE   = 0.5   # +/- 0.5 kyr matching tolerance, matches cv_noise_sensitivity.py
RANDOM_SEED = 42
ALGO = 'KernelCPD'
N_JOBS = -1   # CARC run: use all allocated cores; lower this if running locally

# rbf is ~100-1000x slower per call than every other cost function regardless of
# algorithm -- it gets a smaller seed budget so the grid stays tractable.
# Edit this set to reclassify a cost function as heavy.
HEAVY_COSTS = {'rbf'}
LIGHT_BUDGET = dict(pool_size=20, n_train=14, n_test=6)
HEAVY_BUDGET = dict(pool_size=10, n_train=7, n_test=3)

MIN_SIZES = [5, 7, 10, 15, 20, 30, 40, 50]   # shared default -- override per cost if needed


def logrange(lo, hi, n):
    return list(np.unique(np.round(np.logspace(np.log10(lo), np.log10(hi), n), 3)))

## Section 2 -- Data and ground truth

In [ ]:
def transition_times_from_states(time, states):
    """True transition times (both directions) from a state-label array.

    Mirrors the same helper in cv_noise_sensitivity.py / Noise_Sensitivity_Synthetic_Data.ipynb:
    any stadial/interstadial change counts as ground truth, regardless of direction (Melcher25's
    transitions are symmetric).
    """
    change_idx = np.where(np.diff(states) != 0)[0] + 1
    return time[change_idx]


d = np.load(DATA_PATH)
time_arr   = d['time']
db_noisy   = d['db_noisy']
state_true = d['state_true']
n_seeds    = db_noisy.shape[0]

gt_times = [transition_times_from_states(time_arr, state_true[s]) for s in range(n_seeds)]
n_transitions = [len(g) for g in gt_times]
print(f"Loaded {DATA_PATH}: {n_seeds} seeds, {len(time_arr)} points/series")
print(f"Ground-truth transitions per seed: min={min(n_transitions)}, max={max(n_transitions)}, "
      f"mean={np.mean(n_transitions):.1f}")

## Section 3 -- Detection wrapper and per-combo scoring

`jump` defaults to 1 (no subsampling, ruptures' own default) and only shows up in a param dict
when a cost's cell explicitly passes `jumps=[...]` to `run_one_cost` -- see Section 7.

In [ ]:
def _clean_times(times):
    times = np.asarray(times, dtype=float)
    return times[~np.isnan(times)]


def detect_kernelcpd(time, value, cost, pen, min_size, jump=1):
    series = amt.Series(time=time, value=value, label='grid', verbose=False)
    transitions = series.ruptures(algo=ALGO, cost=cost, pen=pen, min_size=min_size, jump=jump)
    return _clean_times(transitions.jump_times)


def _score_one(params, seed):
    try:
        detected = detect_kernelcpd(time_arr, db_noisy[seed], **params)
    except Exception:
        return dict(precision=0.0, recall=0.0, f1=0.0, false_positives=np.nan)
    m = evaluate_detection(detected=detected, ground_truth=gt_times[seed], tolerance=TOLERANCE)
    return dict(precision=m.precision, recall=m.recall, f1=m.f1_score, false_positives=m.false_positives)

## Section 4 -- Train/test split utilities

Seeds are split ONCE per cost function (deterministic given `RANDOM_SEED`, so rerunning this
notebook reproduces the identical split -- and, since it's the same `RANDOM_SEED` and split logic
as `binseg_grid_red_snr3.ipynb`, the light-budget costs get the identical test seeds there too,
keeping cross-method comparison apples-to-apples). The test half is never touched until each
cost's `run_one_cost` call does its single final evaluation.

In [ ]:
def seed_pool(n_seeds_total, pool_size, random_state=RANDOM_SEED):
    """Fixed subsample of `pool_size` seed indices out of n_seeds_total."""
    rng = np.random.RandomState(random_state)
    return np.sort(rng.choice(n_seeds_total, size=pool_size, replace=False))


def outer_split(pool, n_train, n_test, random_state=RANDOM_SEED):
    """Split `pool` into disjoint train/test seed-index arrays, once."""
    rng = np.random.RandomState(random_state)
    shuffled = rng.permutation(pool)
    train_idx = np.sort(shuffled[:n_train])
    test_idx = np.sort(shuffled[n_train:n_train + n_test])
    return train_idx, test_idx


def split_for_cost(cost, n_seeds_total, random_state=RANDOM_SEED):
    budget = HEAVY_BUDGET if cost in HEAVY_COSTS else LIGHT_BUDGET
    pool = seed_pool(n_seeds_total, budget['pool_size'], random_state=random_state)
    return outer_split(pool, budget['n_train'], budget['n_test'], random_state=random_state)

## Section 5 -- Grid search (train pool) + held-out evaluation (test pool)

Shared machinery every cost's cell calls into -- nothing here is cost-specific, so there's nothing
to copy-paste (and get wrong) per cost.

In [ ]:
def grid_search_train(param_grid, train_idx):
    """Score every (param, seed in train_idx) combo, aggregate per param, pick the
    winner by mean f1 over the train pool only."""
    jobs = [(pi, params, s) for pi, params in enumerate(param_grid) for s in train_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(params, s) for pi, params, s in jobs
    )
    rows = []
    for (pi, params, s), res in zip(jobs, scored):
        row = dict(params)
        row.update(pi=pi, seed=int(s), split='train', **res)
        rows.append(row)
    train_raw = pd.DataFrame(rows)

    param_cols = list(param_grid[0].keys())
    train_summary = (train_raw.groupby(param_cols)
                      .agg(mean_precision=('precision', 'mean'), std_precision=('precision', 'std'),
                           mean_recall=('recall', 'mean'), std_recall=('recall', 'std'),
                           mean_f1=('f1', 'mean'), std_f1=('f1', 'std'),
                           mean_fp=('false_positives', 'mean'))
                      .reset_index())

    best_row = train_summary.loc[train_summary['mean_f1'].idxmax()]
    best_params = {c: best_row[c] for c in param_cols}
    return best_params, best_row, train_summary, train_raw


def evaluate_test(best_params, test_idx):
    """Apply best_params to the untouched-until-now test pool, once."""
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(best_params, s) for s in test_idx
    )
    rows = []
    for s, res in zip(test_idx, scored):
        row = dict(best_params)
        row.update(seed=int(s), split='test', **res)
        rows.append(row)
    test_raw = pd.DataFrame(rows)
    return (test_raw['precision'].mean(), test_raw['recall'].mean(),
            test_raw['f1'].mean(), test_raw['false_positives'].mean(), test_raw)


def pen_sweep_test(cost, pen_grid, min_size, test_idx, extra_params=None):
    """Sweep pen at the winning min_size (and any other winning params, e.g. jump,
    passed via extra_params), test seeds only -- precision/recall tradeoff curve
    for this cost function (the pen-analog of a PR curve)."""
    extra_params = extra_params or {}
    jobs = [(pen, s) for pen in pen_grid for s in test_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(dict(cost=cost, pen=pen, min_size=min_size, **extra_params), s)
        for pen, s in jobs
    )
    rows = []
    for (pen, s), res in zip(jobs, scored):
        rows.append(dict(cost=cost, pen=pen, min_size=min_size, seed=int(s), split='test_sweep', **res))
    sweep_raw = pd.DataFrame(rows)
    sweep_summary = (sweep_raw.groupby('pen')
                      .agg(mean_precision=('precision', 'mean'), mean_recall=('recall', 'mean'),
                           mean_f1=('f1', 'mean'), mean_fp=('false_positives', 'mean'))
                      .reset_index())
    return sweep_summary, sweep_raw


results = {}   # every cost's run_one_cost result lands here, keyed by cost name


def run_one_cost(cost, pens, min_sizes, jumps=None):
    t0 = time.perf_counter()
    train_idx, test_idx = split_for_cost(cost, n_seeds)
    budget = 'HEAVY' if cost in HEAVY_COSTS else 'light'

    if jumps is None:
        param_grid = [{'cost': cost, 'pen': pen, 'min_size': ms}
                      for pen in pens for ms in min_sizes]
    else:
        param_grid = [{'cost': cost, 'pen': pen, 'min_size': ms, 'jump': j}
                      for pen in pens for ms in min_sizes for j in jumps]

    best_params, train_row, train_summary, train_raw = grid_search_train(param_grid, train_idx)
    test_p, test_r, test_f1, test_fp, test_raw = evaluate_test(best_params, test_idx)

    sweep_extra = {'jump': best_params['jump']} if jumps is not None else None
    sweep_summary, sweep_raw = pen_sweep_test(cost, pens, best_params['min_size'], test_idx,
                                               extra_params=sweep_extra)

    on_edge = (best_params['pen'] in (min(pens), max(pens))
               or best_params['min_size'] in (min(min_sizes), max(min_sizes)))

    conclusion_row = dict(
        cost=cost, best_pen=best_params['pen'], best_min_size=best_params['min_size'],
        best_jump=best_params.get('jump', 1),
        train_f1=train_row['mean_f1'], train_precision=train_row['mean_precision'],
        train_recall=train_row['mean_recall'],
        test_f1=test_f1, test_precision=test_p, test_recall=test_r, test_fp=test_fp,
        pen_range=f"{min(pens):.3g}-{max(pens):.3g}",
        min_size_range=f"{min(min_sizes):.0f}-{max(min_sizes):.0f}",
        on_edge=on_edge, seed_budget=budget,
        n_train_seeds=len(train_idx), n_test_seeds=len(test_idx),
        train_seeds=",".join(map(str, train_idx)), test_seeds=",".join(map(str, test_idx)),
    )

    result = dict(train_idx=train_idx, test_idx=test_idx, best_params=best_params,
                  train_summary=train_summary, train_raw=train_raw, test_raw=test_raw,
                  sweep_summary=sweep_summary, sweep_raw=sweep_raw, conclusion_row=conclusion_row)

    elapsed = time.perf_counter() - t0
    edge_note = "  <-- ON GRID EDGE" if on_edge else ""
    print(f"[{budget:5s}] {cost:12s}: pen={best_params['pen']:>8.3g} min_size={best_params['min_size']:>3.0f} "
          f"jump={best_params.get('jump', 1)}  train_f1={train_row['mean_f1']:.3f}  "
          f"test_f1={test_f1:.3f}  ({elapsed:.0f}s){edge_note}")

    results[cost] = result
    return result

## Section 6 -- Plot helpers

Called right after each cost's `run_one_cost` -- immediate feedback before deciding whether to
widen that cost's range and rerun just its own cells.

In [ ]:
def plot_train_surface(cost, result):
    """3D response surface of mean_f1 over the (pen, min_size) TRAIN grid -- this is
    the surface the winning combo was actually picked from."""
    summary = result['train_summary']
    best_pen, best_ms = result['best_params']['pen'], result['best_params']['min_size']

    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]

    pivot = summary.pivot(index='min_size', columns='pen', values='mean_f1')
    pen_vals, ms_vals = pivot.columns.values, pivot.index.values
    X, Y = np.meshgrid(pen_vals, ms_vals)
    Z = pivot.values

    fig = plt.figure(figsize=(6, 5.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='viridis', vmin=0, vmax=np.nanmax(Z),
                     edgecolor='k', linewidth=0.2, antialiased=True)

    zi, xi = list(ms_vals).index(best_ms), list(pen_vals).index(best_pen)
    ax.scatter([best_pen], [best_ms], [Z[zi, xi]], color='red', s=70, marker='*', zorder=10)

    on_edge = result['conclusion_row']['on_edge']
    flag = 'edge -- NOT converged' if on_edge else 'interior -- converged'
    ax.set_title(f"{cost}: train mean_f1\nbest={Z[zi, xi]:.3f} @ pen={best_pen:.3g}, "
                 f"min_size={best_ms:.0f}\n({flag})",
                 fontsize=9, color='#b3261e' if on_edge else '#1a1a1a')
    ax.set_xlabel('pen', fontsize=8)
    ax.set_ylabel('min_size', fontsize=8)
    ax.set_zlabel('mean_f1 (train)', fontsize=8)
    ax.set_zlim(0, max(np.nanmax(Z) * 1.05, 0.05))
    ax.view_init(elev=25, azim=-60)
    fig.suptitle(f"cost='{cost}': TRAIN response surface -- this is what the winner was picked from", fontsize=10)
    plt.tight_layout()
    fig.savefig(f"kernelcpd_grid_red_snr3_{cost}_surface.png", dpi=200, bbox_inches='tight')
    plt.show()


def plot_test_pr(cost, result):
    """Test-set precision/recall tradeoff (pen swept at the winning min_size/jump)."""
    sweep, row = result['sweep_summary'], result['conclusion_row']
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(sweep['mean_recall'], sweep['mean_precision'], color='#2a78d6',
             linewidth=1.5, marker='o', markersize=4, zorder=2)
    ax.scatter([row['test_recall']], [row['test_precision']], color='#e34948',
               s=90, marker='*', zorder=5, label='selected pen')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.set_title(f"{cost}: TEST precision/recall tradeoff")
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

## Section 7 -- Run the grid, one kernel per cell block

`pen`/`min_size` ranges below are starting points, not pre-tuned for KernelCPD specifically (no
prior KernelCPD run exists to seed them from beyond the L2/rbf analogy noted above) -- widen a
kernel's `PENS_*` range and rerun if `on_edge` fires.


### kernel = 'linear'

Equivalent to L2 (mean-shift) cost -- expected to perform comparably to the `l2` rows in the
other notebooks, unlike `cost='linear'` (CostLinear) elsewhere which returned all-zero
detections. `pen` range seeded from the other notebooks' `l2` range.


In [ ]:
PENS_linear = logrange(0.05, 200, 12)
MIN_SIZES_linear = MIN_SIZES

result_linear = run_one_cost('linear', PENS_linear, MIN_SIZES_linear)


In [ ]:
plot_train_surface('linear', result_linear)
plot_test_pr('linear', result_linear)


### kernel = 'rbf' (HEAVY -- ~100-1000x slower per call, smaller seed budget)

`pen` range seeded from BottomUp/Window's `rbf` range (1-100).


In [ ]:
PENS_rbf = logrange(1, 100, 8)
MIN_SIZES_rbf = MIN_SIZES

result_rbf = run_one_cost('rbf', PENS_rbf, MIN_SIZES_rbf)


In [ ]:
plot_train_surface('rbf', result_rbf)
plot_test_pr('rbf', result_rbf)


## Section 8 -- Overview: both kernels' test PR curves together

Run once both kernels above have been run at least once (checks `results` has both -- if you only
reran one, this still works off whatever's currently in `results` from earlier in the kernel
session).


In [ ]:
def plot_test_pr_grid(results, ncols=4):
    costs = list(results.keys())
    nrows = int(np.ceil(len(costs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.2 * nrows), squeeze=False)

    for idx, cost in enumerate(costs):
        ax = axes[idx // ncols][idx % ncols]
        res = results[cost]
        sweep = res['sweep_summary']
        row = res['conclusion_row']

        ax.plot(sweep['mean_recall'], sweep['mean_precision'], color='#2a78d6',
                 linewidth=1.5, marker='o', markersize=3, zorder=2)
        ax.scatter([row['test_recall']], [row['test_precision']], color='#e34948',
                   s=80, marker='*', zorder=5, label='selected pen')

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(cost, fontsize=10)
        ax.set_xlabel('recall', fontsize=7)
        ax.set_ylabel('precision', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.grid(color='#e1e0d9', linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)

    for j in range(len(costs), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')

    fig.subplots_adjust(hspace=0.7, wspace=0.5, top=0.92)
    fig.suptitle("TEST set: precision/recall tradeoff (pen swept at winning min_size, test seeds only)", y=0.98)
    fig.savefig("kernelcpd_grid_red_snr3_test_pr_grid.png", dpi=200, bbox_inches='tight')
    plt.show()


plot_test_pr_grid(results)

## Section 9 -- Save raw / summary / conclusion CSVs

In [ ]:
# raw: every scored (param, seed) row across every cost function, tagged by split --
# 'train' (full grid on the train pool, what selection used), 'test' (the single
# winning combo on the test pool), 'test_sweep' (pen swept at the winning min_size
# on the test pool, for the PR-curve plots above)
raw_frames = []
for cost, result in results.items():
    raw_frames.append(result['train_raw'])
    raw_frames.append(result['test_raw'])
    raw_frames.append(result['sweep_raw'])
raw_df = pd.concat(raw_frames, ignore_index=True, sort=False)
RAW_CSV = "kernelcpd_grid_red_snr3_raw.csv"
raw_df.to_csv(RAW_CSV, index=False)
print(f"Saved {len(raw_df)} rows to {RAW_CSV}")

# summary: the TRAIN grid summary per cost function (what selection was based on) --
# this is what the Section 7 surface plots are drawn from
summary_df = pd.concat([r['train_summary'] for r in results.values()], ignore_index=True, sort=False)
SUMMARY_CSV = "kernelcpd_grid_red_snr3_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {len(summary_df)} rows to {SUMMARY_CSV}")

# conclusion: one row per cost function -- train_f1 (selection score, optimistic)
# next to test_f1 (the honest, held-out score) so the gap between them is visible
conclusion_df = (pd.DataFrame([r['conclusion_row'] for r in results.values()])
                  .sort_values('test_f1', ascending=False).reset_index(drop=True))
CONCLUSION_CSV = "kernelcpd_grid_red_snr3_conclusion.csv"
conclusion_df.to_csv(CONCLUSION_CSV, index=False)
print(f"Saved {len(conclusion_df)} rows to {CONCLUSION_CSV}")

In [ ]:
print("=" * 92)
print("CONCLUSION: KernelCPD on red/snr3 -- train/test split, optimized independently per kernel")
print("=" * 92)
for _, row in conclusion_df.iterrows():
    edge_note = "  <-- ON GRID EDGE: widen this cost's range and rerun" if row['on_edge'] else ""
    gap = row['train_f1'] - row['test_f1']
    print(f"{row['cost']:>12s} [{row['seed_budget']:5s}]: pen={row['best_pen']:>8.3g}  min_size={row['best_min_size']:>3.0f}  "
          f"train_f1={row['train_f1']:.3f}  test_f1={row['test_f1']:.3f}  (gap={gap:+.3f}){edge_note}")

n_edge = int(conclusion_df['on_edge'].sum())
if n_edge:
    print(f"\n{n_edge} kernel(s) did NOT converge on the train grid (optimum on the tested "
          f"boundary) -- widen those ranges and rerun before trusting their numbers.")
else:
    print(f"\nAll {len(conclusion_df)} kernels converged (train optimum interior to their tested range).")

best_overall = conclusion_df.iloc[0]
print(f"\nBest kernel by TEST f1: '{best_overall['cost']}' "
      f"(pen={best_overall['best_pen']:.3g}, min_size={best_overall['best_min_size']:.0f}, "
      f"test_f1={best_overall['test_f1']:.3f})")